# Baseline SLM Evaluation: Qwen2.5-1.5B-Instruct

**Project:** Python Learning Assistant Using Small Language Model (SLM)  
**Notebook:** `notebooks/baseline_evaluation.ipynb`  
**Target Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Unadapted Pretrained Baseline)  
**Pinned Revision:** `989aa7980e4cf806f80c7fef2b1adb7bc71aa306` (Apache-2.0)  
**Evaluation Dataset:** `data/processed/test.jsonl` (38 Held-Out Test Questions)  
**Target Hardware:** Google Colab FREE Tier GPU (NVIDIA Tesla T4 or any available CUDA GPU). Zero paid APIs or subscriptions required.  
**Output Artifact:** `experiments/baseline_results_actual.json` (prompted for one-click browser download upon completion)  

---
### Step-by-Step Instructions for Google Colab:

1. **Select GPU Runtime:**
   - Navigate to: **Runtime -> Change runtime type**.
   - Set **Hardware accelerator** to **T4 GPU** (Standard Free tier).

2. **Workspace & Dataset Setup (Recommended):**
   - If not already done, clone the repository and switch directory:
     ```bash
     !git clone https://github.com/sasikalascgm-droid/python-learning-assistant-slm.git
     %cd python-learning-assistant-slm
     ```
   - Note: Step 5 includes automatic workspace detection and GitHub fallback recovery so the 38 test cases load reliably regardless of execution folder.

3. **Execution:**
   - Click **Runtime -> Run all**.
   - Step 5 evaluates all 38 test questions.
   - Step 6 compiles latency, throughput, token counts, and exports `experiments/baseline_results_actual.json`.


In [ ]:

# Step 1: Runtime Hardware Audit & Colab Workspace Configuration

import os
import sys
import psutil
import torch
from pathlib import Path

# Auto-detect Google Colab
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

# Stop accidental local execution
if not IN_COLAB:
    raise RuntimeError(
        "Actual baseline evaluation must run on Google Colab "
        "with a CUDA GPU. Do not load the full model on this local laptop."
    )

# Set working directory if repository is already cloned
for repo_candidate in [
    '/content/python-learning-assistant-slm',
    '/content/slm-python-learning-assistant'
]:
    if (Path(repo_candidate) / 'data/processed/test.jsonl').exists():
        os.chdir(repo_candidate)
        print(f'Switched working directory to: {repo_candidate}')
        break

print('=== Runtime Hardware Audit ===')

# Require CUDA GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Please select a GPU runtime "
        "in Google Colab (Runtime > Change runtime type > T4 GPU)."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f'CUDA GPU Detected : {gpu_name}')
print(f'Available VRAM    : {vram_gb:.2f} GB')

device = 'cuda'

# Select supported precision
major, _ = torch.cuda.get_device_capability(0)
dtype = (
    torch.bfloat16
    if major >= 8 and torch.cuda.is_bf16_supported()
    else torch.float16
)

print(f'Active Device: {device}, Precision: {dtype}')


In [ ]:
# Step 2: Install Verified Open-Source Dependencies (Pinning huggingface-hub<2.0 for zero conflict)
!pip install -q "transformers>=5.22.0" "huggingface-hub>=1.31.0,<2.0" "accelerate>=1.15.0" psutil pyyaml


In [ ]:
# Step 2b: Package Compatibility Verification & Conflict Check
import sys
import subprocess

print('=== Installed Package Versions ===')
for pkg in ['torch', 'transformers', 'huggingface_hub', 'accelerate']:
    try:
        mod = __import__(pkg)
        print(f'{pkg:16}: {getattr(mod, "__version__", "unknown")}')
    except ImportError:
        print(f'{pkg:16}: not installed')

print('\n=== Dependency Conflict Audit (pip check) ===')
check_res = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
if check_res.returncode == 0:
    print('PASSED: Zero dependency conflicts detected across installed packages.')
else:
    print(f'Notice from pip check:\n{check_res.stdout.strip()}')


In [ ]:
# Step 3: Load Base Model and Tokenizer (Singleton Initialization with Pinned Revision)
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
PINNED_REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
print(f'Loading tokenizer and model for "{MODEL_ID}" (revision {PINNED_REVISION})...')

load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=PINNED_REVISION, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=PINNED_REVISION,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)
if device == 'cuda':
    model = model.to('cuda')
model.eval()
model.config.use_cache = True

load_time = time.perf_counter() - load_start
print(f'Model successfully loaded in {load_time:.2f} seconds.')


In [ ]:
# Step 4: Define System Prompt and Inference Pipeline
SYSTEM_PROMPT = '''You are an expert, patient, and friendly Python Learning Assistant.
Your mission is to help beginner students master Python programming.
Follow these principles:
1. Explain concepts in clear, simple English without unnecessary jargon.
2. Provide idiomatic, clean, and bug-free Python code examples.
3. Break down code step-by-step or line-by-line when explaining snippets.
4. When diagnosing errors, clearly explain the cause and provide the exact fix.
5. Always encourage the student with beginner-friendly practice questions and hints.
'''

def generate_slm_response(instruction, input_text='', max_new_tokens=512, temperature=0.7, top_p=0.9, repetition_penalty=1.1):
    user_content = instruction.strip()
    if input_text and input_text.strip():
        user_content += f'\n\n```python\n{input_text.strip()}\n```'

    # Construct messages and prefer tokenizer.apply_chat_template if supported
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
        {'role': 'user', 'content': user_content}
    ]

    if hasattr(tokenizer, 'apply_chat_template') and getattr(tokenizer, 'chat_template', None):
        try:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = (
                f'<|im_start|>system\n{SYSTEM_PROMPT.strip()}<|im_end|>\n'
                f'<|im_start|>user\n{user_content}<|im_end|>\n'
                f'<|im_start|>assistant\n'
            )
    else:
        prompt = (
            f'<|im_start|>system\n{SYSTEM_PROMPT.strip()}<|im_end|>\n'
            f'<|im_start|>user\n{user_content}<|im_end|>\n'
            f'<|im_start|>assistant\n'
        )

    inputs = tokenizer(prompt, return_tensors='pt')
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

    input_tokens = inputs['input_ids'].shape[1]

    # Configure generation kwargs; pass temperature and top_p only when sampling is enabled
    do_sample = bool(temperature and temperature > 0.0)
    gen_kwargs = {
        'max_new_tokens': max_new_tokens,
        'repetition_penalty': repetition_penalty,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'use_cache': True
    }
    if do_sample:
        gen_kwargs['temperature'] = temperature
        gen_kwargs['top_p'] = top_p

    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    if device == 'cuda':
        torch.cuda.synchronize()
    latency = time.perf_counter() - t0

    gen_tokens = outputs[0][input_tokens:]
    out_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    tok_per_sec = len(gen_tokens) / max(latency, 0.001)

    return {
        'response': out_text,
        'latency_seconds': round(latency, 3),
        'input_tokens': input_tokens,
        'output_tokens': len(gen_tokens),
        'tokens_per_second': round(tok_per_sec, 2)
    }


In [ ]:
# Step 5: Execute Evaluation Across Held-Out Test Set (38 cases)
import json
import os
import urllib.request
from pathlib import Path

# Multi-tier candidate resolution for Google Colab and local environments
candidate_paths = [
    Path('data/processed/test.jsonl'),
    Path('data/test.jsonl'),
    Path('/content/python-learning-assistant-slm/data/processed/test.jsonl'),
    Path('/content/python-learning-assistant-slm/data/test.jsonl'),
    Path('/content/slm-python-learning-assistant/data/processed/test.jsonl'),
    Path('/content/slm-python-learning-assistant/data/test.jsonl'),
]

test_file = None
for p in candidate_paths:
    if p.exists() and p.is_file():
        test_file = p
        break

# Auto-recovery for Google Colab if executing in /content without cloned repo
if test_file is None:
    print('Held-out test file not found in local paths. Attempting Colab workspace recovery...')
    # 1. Clone repository if needed and ALWAYS checkout the pinned revision
    import subprocess

    colab_repo_dir = Path('/content/python-learning-assistant-slm')
    pinned_commit = 'd7173d5a9801ca4115607260ae1b3d743f588d6d'

    try:
        if not colab_repo_dir.exists():
            print('Cloning repository into /content/python-learning-assistant-slm...')
            subprocess.run(
                [
                    'git', 'clone',
                    'https://github.com/sasikalascgm-droid/python-learning-assistant-slm.git',
                    str(colab_repo_dir)
                ],
                check=True
            )

        # Always checkout the exact pinned commit
        subprocess.run(
            [
                'git', '-C', str(colab_repo_dir),
                'checkout', pinned_commit
            ],
            check=True
        )

        if (colab_repo_dir / 'data/processed/test.jsonl').exists():
            test_file = colab_repo_dir / 'data/processed/test.jsonl'
            os.chdir(colab_repo_dir)
            print(f'Switched working directory to: {colab_repo_dir}')

    except Exception as e:
        print(f'Git repository recovery encountered: {e}')
    
    # 2. Direct GitHub download fallback if clone was not possible
    if test_file is None or not test_file.exists():
        fallback_dir = Path('data/processed')
        fallback_dir.mkdir(parents=True, exist_ok=True)
        test_file = fallback_dir / 'test.jsonl'
        raw_url = 'https://raw.githubusercontent.com/sasikalascgm-droid/python-learning-assistant-slm/d7173d5a9801ca4115607260ae1b3d743f588d6d/data/processed/test.jsonl'
        print(f'Downloading official held-out test.jsonl from GitHub: {raw_url}...')
        urllib.request.urlretrieve(raw_url, test_file)
        print(f'Successfully downloaded test dataset to: {test_file}')

if not test_file or not test_file.exists():
    raise FileNotFoundError(
       f'FATAL: Held-out test dataset could not be located. Searched candidates: {[str(p) for p in candidate_paths]}'
    )

print(f'Loading held-out test cases from: {test_file.resolve()}')

test_cases = []
with open(test_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            test_cases.append(json.loads(line))

# Validate that exactly 38 test cases are loaded
assert len(test_cases) == 38, f'Expected exactly 38 held-out test cases, but found {len(test_cases)}!'

# Validate schema and types for each test case
for idx, tc in enumerate(test_cases, 1):
    if not isinstance(tc, dict):
        raise ValueError(f'Test case {idx} is not a valid JSON dictionary.')
    for key in ('instruction', 'input', 'output'):
        if key not in tc:
            raise ValueError(f"Test case {idx} missing mandatory field: '{key}'")
    if not isinstance(tc['instruction'], str) or not tc['instruction'].strip():
        raise ValueError(f"Test case {idx} 'instruction' must be a non-empty string.")
    if not isinstance(tc['output'], str) or not tc['output'].strip():
        raise ValueError(f"Test case {idx} 'output' must be a non-empty string.")
    if not isinstance(tc['input'], str):
        raise ValueError(f"Test case {idx} 'input' must be a string (can be empty).")

print(f'Successfully loaded and validated {len(test_cases)} held-out test cases (test set strictly verified).')

results = []

for idx, tc in enumerate(test_cases, 1):
    inst = tc['instruction']
    inp = tc.get('input', '')
    gt = tc.get('output', '')

    print(f'[{idx}/{len(test_cases)}] Evaluating: {inst[:45]}...')

    try:
        out = generate_slm_response(inst, inp)

        if not out.get('response') or not out['response'].strip():
            raise ValueError('Model generated an empty response.')

        results.append({
            'test_case_id': idx,
            'instruction': inst,
            'input': inp,
            'ground_truth': gt,
            'model_response': out['response'],
            'latency_seconds': out['latency_seconds'],
            'input_tokens': out['input_tokens'],
            'output_tokens': out['output_tokens'],
            'tokens_per_second': out['tokens_per_second'],
            'status': 'success',
            'error': None
        })

    except Exception as e:
        print(f'Failed test case {idx}: {e}')

        results.append({
            'test_case_id': idx,
            'instruction': inst,
            'input': inp,
            'ground_truth': gt,
            'model_response': '',
            'latency_seconds': 0,
            'input_tokens': 0,
            'output_tokens': 0,
            'tokens_per_second': 0,
            'status': 'failed',
            'error': str(e)
        })

print('Evaluation complete!')
print('Successful:', sum(r['status'] == 'success' for r in results))
print('Failed:', sum(r['status'] == 'failed' for r in results))


In [ ]:
# Step 6: Generate Baseline Summary and Save Actual Results

import time
import json
import math
import statistics
from pathlib import Path

successful_results = [
    r for r in results if r['status'] == 'success'
]

latencies = [r['latency_seconds'] for r in successful_results]
tokens = [r['output_tokens'] for r in successful_results]
tps = [r['tokens_per_second'] for r in successful_results]

successful_cases = len(successful_results)
failed_cases = len(results) - successful_cases

tot_lat = sum(latencies)
tot_tok = sum(tokens)

expected_test_cases = 38
evaluation_complete = len(results) == expected_test_cases

# 1. Hardware & Environment Verifications
cuda_available = bool(torch.cuda.is_available())
device_is_cuda = bool(device == 'cuda')

# 2. Safely verify that all model parameters are allocated on CUDA
try:
    model_parameters_on_cuda = bool(
        'model' in globals()
        and model is not None
        and hasattr(model, 'parameters')
        and any(p.is_cuda for p in model.parameters())
        and all(p.is_cuda for p in model.parameters())
    )
except Exception:
    model_parameters_on_cuda = False

# 3. Model ID and Pinned Revision Verifications
model_id_verified = bool(MODEL_ID == 'Qwen/Qwen2.5-1.5B-Instruct')
revision_verified = bool(PINNED_REVISION == '989aa7980e4cf806f80c7fef2b1adb7bc71aa306')

# 4. FP16 Precision Verification for Tesla T4 runtime
try:
    is_fp16 = bool(
        dtype == torch.float16
        or str(dtype).lower() in ('torch.float16', 'float16')
    )
    if cuda_available:
        major, _ = torch.cuda.get_device_capability(0)
        # On SM < 8.0 (Tesla T4 is SM 7.5), FP16 is strictly required
        if major < 8:
            fp16_precision_verified = is_fp16
        else:
            # On SM 8.0+ (Ampere/Hopper), BF16 or FP16 is accepted
            fp16_precision_verified = bool(
                is_fp16
                or dtype == torch.bfloat16
                or str(dtype).lower() in ('torch.bfloat16', 'bfloat16')
            )
    else:
        fp16_precision_verified = False
except Exception:
    fp16_precision_verified = False

# 5. Model Load Time Verification
try:
    load_time_verified = bool(
        'load_time' in globals()
        and isinstance(load_time, (int, float))
        and not math.isnan(load_time)
        and not math.isinf(load_time)
        and load_time >= 0
    )
except Exception:
    load_time_verified = False

# 6. Test Case IDs and Count Verifications
expected_case_ids = set(range(1, expected_test_cases + 1))
actual_case_ids = [r.get('test_case_id') for r in results]

test_count_verified = bool(len(results) == expected_test_cases)
test_ids_verified = bool(
    len(actual_case_ids) == expected_test_cases
    and len(set(actual_case_ids)) == expected_test_cases
    and set(actual_case_ids) == expected_case_ids
)
success_count_verified = bool(successful_cases == expected_test_cases)
zero_failures_verified = bool(failed_cases == 0)

all_verifications_passed = bool(
    cuda_available
    and device_is_cuda
    and model_parameters_on_cuda
    and model_id_verified
    and revision_verified
    and fp16_precision_verified
    and load_time_verified
    and test_count_verified
    and test_ids_verified
    and success_count_verified
    and zero_failures_verified
)

# Determine run status with explicit non-success categorization
if all_verifications_passed:
    run_status = 'actual_gpu_run_complete'
elif not cuda_available:
    run_status = 'cpu_run_not_valid_for_baseline'
elif not device_is_cuda:
    run_status = 'device_not_cuda'
elif not model_parameters_on_cuda:
    run_status = 'model_not_on_cuda'
elif not model_id_verified:
    run_status = 'model_id_mismatch'
elif not revision_verified:
    run_status = 'model_revision_mismatch'
elif not fp16_precision_verified:
    run_status = 'precision_mismatch_for_runtime'
elif not load_time_verified:
    run_status = 'invalid_load_time'
elif not (test_count_verified and test_ids_verified):
    run_status = 'incomplete_gpu_run'
elif not (success_count_verified and zero_failures_verified):
    run_status = 'actual_gpu_run_with_failures'
else:
    run_status = 'verification_failed'

verifications = {
    'torch_cuda_available': cuda_available,
    'device_is_cuda': device_is_cuda,
    'model_parameters_on_cuda': model_parameters_on_cuda,
    'model_id_verified': model_id_verified,
    'pinned_revision_verified': revision_verified,
    'fp16_precision_verified': fp16_precision_verified,
    'load_time_verified': load_time_verified,
    'expected_test_cases_evaluated': test_count_verified,
    'test_case_ids_verified': test_ids_verified,
    'all_test_cases_successful': success_count_verified,
    'zero_failures': zero_failures_verified,
    'all_verifications_passed': all_verifications_passed,
}

summary = {
    'evaluation_name': 'Qwen2.5-1.5B-Instruct Baseline Evaluation',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model_id': MODEL_ID,
    'model_revision': PINNED_REVISION,
    'device': device,
    'precision': str(dtype),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_vram_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else None,
    'model_load_time_seconds': round(load_time, 2) if load_time_verified else None,
    'run_status': run_status,
    'evaluation_complete': evaluation_complete,
    'expected_test_cases': expected_test_cases,
    'verifications': verifications,
    'total_test_cases': len(results),
    'successful_cases': successful_cases,
    'failed_cases': failed_cases,
    'total_tokens_generated': tot_tok,
    'total_generation_time_seconds': round(tot_lat, 2),
    'true_aggregate_throughput_tok_per_sec': round(tot_tok / max(tot_lat, 0.001), 2),
    'average_latency_seconds': round(tot_lat / len(latencies), 3) if latencies else None,
    'median_latency_seconds': round(statistics.median(latencies), 3) if latencies else None,
    'average_tokens_generated': round(tot_tok / len(tokens), 1) if tokens else None,
    'average_tokens_per_second': round(sum(tps) / len(tps), 2) if tps else None,
    'test_results': results
}

out_dir = Path('experiments')
out_dir.mkdir(parents=True, exist_ok=True)

results_file = out_dir / 'baseline_results_actual.json'

with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\n=== Baseline Evaluation Summary ===')
print(f"Run status: {run_status}")
print(f"All verifications passed: {all_verifications_passed}")
print(f"Total test cases: {len(results)}")
print(f"Successful cases: {successful_cases}")
print(f"Failed cases: {failed_cases}")
print(f"Results saved to: {results_file.resolve()}")
print(f"Download file: experiments/baseline_results_actual.json")